In [1]:
import importlib
import sys


def reimport_trackers():
    """Force-reload all trackers submodules already in sys.modules."""
    mods = [k for k in sys.modules if k.startswith("trackers")]
    for m in sorted(mods, reverse=True):
        importlib.reload(sys.modules[m])
    print(f"Reloaded {len(mods)} trackers module(s).")

# Re-ID Evaluation — OSNet on Market-1501 and MSMT17

This notebook reproduces the standard **same-domain** re-ID benchmark numbers for OSNet x1.0.
Each dataset is evaluated with the checkpoint that was *trained on that same dataset*
(from the [torchreid model zoo](https://kaiyangzhou.github.io/deep-person-reid/MODEL_ZOO)),
which is the only fair comparison against published numbers.

1. **Market-1501** (~1.3 GB, ~10 min on T4) — Expected: ~94.2 Rank-1 / 82.6 mAP.
2. **MSMT17** (~4 GB, ~40 min on T4) — Expected: ~74.9 Rank-1 / 43.8 mAP.

> **Note:** the default `ReIDModel.from_pretrained()` checkpoint is OSNet trained on
> MSMT17 with `combineall=True` (train+test combined). Evaluating *that* checkpoint on
> MSMT17 leaks the test set (near-100% scores) and on Market-1501 measures cross-domain
> transfer (~61 R1). To reproduce paper numbers we instead load each dataset's own
> same-domain checkpoint below.

> **Runtime:** `Runtime → Change runtime type → T4 GPU` before running.

---

## 1. Install

In [2]:
# Install the trackers library from the feature branch + reid optional deps.
# The [reid] extra pulls in torch, torchvision, timm, and huggingface-hub.
GIT_REF = "git+https://github.com/roboflow/trackers.git@feat/core/reid"

!pip install -q --upgrade pip
# Colab ships CUDA torch — avoid trackers[reid] (pulls a conflicting PyPI torch).
!pip install -q timm huggingface-hub safetensors gdown
!pip install -q --no-cache-dir --force-reinstall --no-deps "trackers @ {GIT_REF}"
!pip install -q supervision scipy opencv-python rich requests pydeprecate

reimport_trackers()

import trackers
print(f"trackers from {trackers.__file__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.9 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reloaded 0 trackers module(s).


In [3]:
# Confirm GPU is available.
import torch

print(
    f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  |  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}"
)

PyTorch 2.11.0+cu128  |  CUDA: True  |  Device: Tesla T4


## 2. Setup

Imports the re-ID API and defines a small helper to fetch the per-dataset OSNet x1.0
checkpoints from the torchreid model zoo (hosted on Google Drive, ~9 MB each).
The matching checkpoint is loaded inside each dataset's section below via
`ReIDModel.from_pretrained(path, architecture="osnet_x1_0")`.

In [4]:
!pip install -q gdown
import os

import gdown

from trackers.core.reid import ReidEvaluator, ReIDModel

CKPT_DIR = "/content/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)


def download_checkpoint(gdrive_id: str, filename: str) -> str:
    """Download an OSNet checkpoint from the torchreid model zoo (Google Drive)."""
    path = os.path.join(CKPT_DIR, filename)
    if not os.path.exists(path):
        gdown.download(id=gdrive_id, output=path, quiet=False)
    return path


def print_comparison(name, cos, euc, zoo_r1, zoo_map):
    """Print a cosine-vs-euclidean comparison table against model-zoo targets."""
    print(f"\n{name} — distance metric comparison")
    print(f"{'metric':<10}{'cosine':>10}{'euclidean':>12}{'model zoo':>12}")
    print("-" * 44)
    print(f"{'Rank-1':<10}{cos.rank1:>9.1f}%{euc.rank1:>11.1f}%{zoo_r1:>11.1f}%")
    print(f"{'mAP':<10}{cos.map:>9.1f}%{euc.map:>11.1f}%{zoo_map:>11.1f}%")
    print(f"{'Rank-5':<10}{cos.rank5:>9.1f}%{euc.rank5:>11.1f}%{'—':>12}")
    print(f"{'Rank-10':<10}{cos.rank10:>9.1f}%{euc.rank10:>11.1f}%{'—':>12}")
    print(f"{'mINP':<10}{cos.minp:>9.1f}%{euc.minp:>11.1f}%{'—':>12}")


print("Setup complete.")

Setup complete.


---
## 3. Market-1501

**Expected numbers (OSNet x1.0 trained on Market-1501, torchreid model zoo):** Rank-1 ≈ 94.2 %  |  mAP ≈ 82.6 %

The model-zoo numbers use **raw Euclidean** distance. We report both that and our
default **cosine** (L2-normalised) distance so the difference is visible.

Market-1501 is ~1.3 GB. We download it via gdown (official Google Drive mirror).
If gdown fails, see the alternative download cell below.

In [5]:
!pip install -q gdown
import os
import zipfile

MARKET_ZIP = "/content/Market-1501.zip"
MARKET_DIR = "/content/Market-1501-v15.09.15"

if not os.path.exists(MARKET_DIR):
    # Google Drive file ID for Market-1501-v15.09.15.zip
    gdown.download(id="0B8-rUzbwVRk0c054eEozWG9COHM", output=MARKET_ZIP, quiet=False)
    with zipfile.ZipFile(MARKET_ZIP, "r") as zf:
        zf.extractall("/content")
    print("Extracted to", MARKET_DIR)
else:
    print("Already downloaded.")

Downloading...
From (original): https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM
From (redirected): https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM&confirm=t&uuid=c6f4366e-f9bb-4c8f-85b7-2e3b1c773c8e
To: /content/Market-1501.zip
100%|██████████| 153M/153M [00:01<00:00, 90.1MB/s] 


Extracted to /content/Market-1501-v15.09.15


In [6]:
# Alternative download if gdown fails (paste the unzip path that matches your mirror):
# !wget -q -O /content/Market-1501.zip "<YOUR_MIRROR_URL>"
# !unzip -q /content/Market-1501.zip -d /content

In [7]:
from trackers.core.reid import load_market1501

query_m, gallery_m = load_market1501(MARKET_DIR)
print(f"Market-1501 — query: {len(query_m):,}  |  gallery: {len(gallery_m):,}")

Market-1501 — query: 3,368  |  gallery: 19,732


In [8]:
# Load the OSNet x1.0 checkpoint trained on Market-1501 (torchreid model zoo).
market_ckpt = download_checkpoint("1vduhq5DpN2q1g4fYEZfPI17MJeh9qyrA", "osnet_x1_0_market1501.pth")
model_market = ReIDModel.from_pretrained(market_ckpt, architecture="osnet_x1_0")
evaluator_market = ReidEvaluator(model_market, batch_size=256)

# Cosine distance on L2-normalised embeddings (our default) — extracts once.
result_market = evaluator_market.evaluate(query_m, gallery_m, distance="cosine", return_distmat=False)
# Raw Euclidean distance (the torchreid model-zoo protocol) — reuse the same
# embeddings and just re-score, so we don't pay for extraction twice.
result_market_euc = evaluator_market.evaluate(
    query_m,
    gallery_m,
    distance="euclidean",
    return_distmat=False,
    query_embeddings=result_market.query_embeddings,
    gallery_embeddings=result_market.gallery_embeddings,
)

print_comparison("Market-1501", result_market.metrics, result_market_euc.metrics, 94.2, 82.6)

Downloading...
From: https://drive.google.com/uc?id=1vduhq5DpN2q1g4fYEZfPI17MJeh9qyrA
To: /content/checkpoints/osnet_x1_0_market1501.pth
100%|██████████| 10.4M/10.4M [00:00<00:00, 30.6MB/s]


Extracting query embeddings  (3368 images)…
Extracting gallery embeddings (19732 images)…
Computing distance matrix (cosine)…
Computing metrics…

Results (cosine)
--------------------------------------------------
mAP: 83.6%  Rank-1: 94.2%  Rank-5: 97.8%  Rank-10: 98.6%  mINP: 55.8%  (n=3368)
--------------------------------------------------
Computing distance matrix (euclidean)…
Computing metrics…

Results (euclidean)
--------------------------------------------------
mAP: 82.5%  Rank-1: 94.4%  Rank-5: 97.8%  Rank-10: 98.6%  mINP: 51.0%  (n=3368)
--------------------------------------------------

Market-1501 — distance metric comparison
metric        cosine   euclidean   model zoo
--------------------------------------------
Rank-1         94.2%       94.4%       94.2%
mAP            83.6%       82.5%       82.6%
Rank-5         97.8%       97.8%           —
Rank-10        98.6%       98.6%           —
mINP           55.8%       51.0%           —


---
## 4. MSMT17

**Expected numbers (OSNet x1.0 trained on MSMT17, torchreid model zoo):** Rank-1 ≈ 74.9 %  |  mAP ≈ 43.8 % (raw Euclidean; we also report cosine)

### Option A — download directly in Colab (recommended)

Downloads `MSMT17_V1.zip` (~2.56 GB) from the community Hugging Face mirror
[`xianpeijie/MSMT17_V1`](https://huggingface.co/datasets/xianpeijie/MSMT17_V1)
using `huggingface_hub` (already installed as part of the `[reid]` extra).

In [9]:
# Option A: download from Hugging Face community mirror (~2.56 GB)
import os
import zipfile

from huggingface_hub import hf_hub_download

MSMT17_DIR = "/content/MSMT17_V1"

if not os.path.exists(MSMT17_DIR):
    msmt17_zip = hf_hub_download(
        repo_id="xianpeijie/MSMT17_V1",
        filename="MSMT17_V1.zip",
        repo_type="dataset",
        local_dir="/content",
    )
    print("Extracting MSMT17 (~2.56 GB, may take a few minutes)…")
    with zipfile.ZipFile(msmt17_zip, "r") as zf:
        zf.extractall("/content")
    print("Done →", MSMT17_DIR)
else:
    print("Already extracted.")

Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).


MSMT17_V1.zip:   0%|          | 0.00/2.56G [00:00<?, ?B/s]

Extracting MSMT17 (~2.56 GB, may take a few minutes)…
Done → /content/MSMT17_V1


In [10]:
from trackers.core.reid import load_msmt17

query_ms, gallery_ms = load_msmt17(MSMT17_DIR)
print(f"MSMT17 — query: {len(query_ms):,}  |  gallery: {len(gallery_ms):,}")

MSMT17 — query: 11,659  |  gallery: 82,161


In [11]:
# Load the OSNet x1.0 checkpoint trained on MSMT17 (standard split, torchreid model zoo).
# Note: use from_pretrained with a bare checkpoint path and the architecture name.
# Do NOT use the default alias (osnet_x1_0_msmt17_combineall) to benchmark MSMT17 —
# it trains on the MSMT17 test identities, so scores would be inflated.
msmt17_ckpt = download_checkpoint("112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M", "osnet_x1_0_msmt17.pth")
model_msmt17 = ReIDModel.from_pretrained(msmt17_ckpt, architecture="osnet_x1_0")
evaluator_msmt17 = ReidEvaluator(model_msmt17, batch_size=256)

# Cosine distance on L2-normalised embeddings (our default) — extracts once.
result_msmt17 = evaluator_msmt17.evaluate(query_ms, gallery_ms, distance="cosine", return_distmat=False)
# Raw Euclidean distance (the torchreid model-zoo protocol) — reuse embeddings.
result_msmt17_euc = evaluator_msmt17.evaluate(
    query_ms,
    gallery_ms,
    distance="euclidean",
    return_distmat=False,
    query_embeddings=result_msmt17.query_embeddings,
    gallery_embeddings=result_msmt17.gallery_embeddings,
)

print_comparison("MSMT17", result_msmt17.metrics, result_msmt17_euc.metrics, 74.9, 43.8)

Downloading...
From: https://drive.google.com/uc?id=112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M
To: /content/checkpoints/osnet_x1_0_msmt17.pth
100%|██████████| 11.0M/11.0M [00:01<00:00, 9.45MB/s]


Extracting query embeddings  (11659 images)…
Extracting gallery embeddings (82161 images)…
Computing distance matrix (cosine)…
Computing metrics…

Results (cosine)
--------------------------------------------------
mAP: 47.8%  Rank-1: 75.9%  Rank-5: 86.1%  Rank-10: 89.3%  mINP: 8.6%  (n=11659)
--------------------------------------------------
Computing distance matrix (euclidean)…
Computing metrics…

Results (euclidean)
--------------------------------------------------
mAP: 43.7%  Rank-1: 74.8%  Rank-5: 85.1%  Rank-10: 88.5%  mINP: 5.3%  (n=11659)
--------------------------------------------------

MSMT17 — distance metric comparison
metric        cosine   euclidean   model zoo
--------------------------------------------
Rank-1         75.9%       74.8%       74.9%
mAP            47.8%       43.7%       43.8%
Rank-5         86.1%       85.1%           —
Rank-10        89.3%       88.5%           —
mINP            8.6%        5.3%           —


---
## 5. BoT-SORT FastReID encoder (MOT17 SBS-S50)

The [BoT-SORT model zoo](https://github.com/niraharon/bot-sort#model-zoo) publishes a **FastReID Strong Baseline** checkpoint trained on MOT17 train-half GT crops (`mot17_sbs_S50.pth`). This is the encoder used in the paper's BoT-SORT-ReID ablation.

We load it via the curated alias `fastreid_mot17_sbs50` (ResNeSt50 + GeM + BNNeck, **384×128** input). Run **§4 MSMT17** first so `query_ms` / `gallery_ms` are in memory — the cell below measures **cross-domain** retrieval (MOT-trained encoder on MSMT17), which is expected to score lower than same-domain OSNet numbers in §4.

In [12]:
# Load BoT-SORT / FastReID MOT17 SBS-S50 (~318 MB from Google Drive).
fastreid_ckpt = download_checkpoint("1QZFWpoa80rqo7O-HXmlss8J8CnS7IUsN", "mot17_sbs_S50.pth")

# Option A: curated alias (uses gd:// cache on repeat runs)
model_fastreid = ReIDModel.from_pretrained("fastreid_mot17_sbs50")

# Option B: explicit local path (equivalent)
# model_fastreid = ReIDModel.from_pretrained(
#     fastreid_ckpt,
#     architecture="fastreid_sbs_resnest50",
# )

print(model_fastreid.preprocessing.describe())

Downloading...
From (original): https://drive.google.com/uc?id=1QZFWpoa80rqo7O-HXmlss8J8CnS7IUsN
From (redirected): https://drive.google.com/uc?id=1QZFWpoa80rqo7O-HXmlss8J8CnS7IUsN&confirm=t&uuid=42efb72a-b461-46fe-b44d-0505db151ad1
To: /content/checkpoints/mot17_sbs_S50.pth
100%|██████████| 318M/318M [00:04<00:00, 75.1MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1QZFWpoa80rqo7O-HXmlss8J8CnS7IUsN
From (redirected): https://drive.google.com/uc?id=1QZFWpoa80rqo7O-HXmlss8J8CnS7IUsN&confirm=t&uuid=f904a360-f3b0-42bc-9ab4-b4c85893fb03
To: /root/.cache/trackers/weights/mot17_sbs_S50.pth
100%|██████████| 318M/318M [00:03<00:00, 89.1MB/s] 


ReIDPreprocessing(resize=384x128 [bilinear], BGR→RGB, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), embed_norm=L2)


In [14]:
# Cross-domain gallery retrieval: MOT17-trained encoder on MSMT17 query/gallery.
# Requires §4 MSMT17 cells (query_ms, gallery_ms) to have been run.
evaluator_fastreid = ReidEvaluator(model_fastreid, batch_size=256)

result_fastreid_msmt17_cos = evaluator_fastreid.evaluate(
    query_ms, gallery_ms, distance="cosine", return_distmat=False
)
result_fastreid_msmt17_euc = evaluator_fastreid.evaluate(
    query_ms,
    gallery_ms,
    distance="euclidean",
    return_distmat=False,
    query_embeddings=result_fastreid_msmt17_cos.query_embeddings,
    gallery_embeddings=result_fastreid_msmt17_cos.gallery_embeddings,
)

print("FastReID MOT17 SBS50 on MSMT17 (cross-domain)")
print_comparison(
    "MSMT17 (FastReID MOT17)",
    result_fastreid_msmt17_cos.metrics,
    result_fastreid_msmt17_euc.metrics,
    float("nan"),
    float("nan"),
)

Extracting query embeddings  (11659 images)…
Extracting gallery embeddings (82161 images)…
Computing distance matrix (cosine)…
Computing metrics…

Results (cosine)
--------------------------------------------------
mAP: 2.1%  Rank-1: 8.7%  Rank-5: 15.7%  Rank-10: 19.8%  mINP: 0.1%  (n=11659)
--------------------------------------------------
Computing distance matrix (euclidean)…
Computing metrics…

Results (euclidean)
--------------------------------------------------
mAP: 1.8%  Rank-1: 8.1%  Rank-5: 14.8%  Rank-10: 18.7%  mINP: 0.1%  (n=11659)
--------------------------------------------------
FastReID MOT17 SBS50 on MSMT17 (cross-domain)

MSMT17 (FastReID MOT17) — distance metric comparison
metric        cosine   euclidean   model zoo
--------------------------------------------
Rank-1          8.7%        8.1%        nan%
mAP             2.1%        1.8%        nan%
Rank-5         15.7%       14.8%           —
Rank-10        19.8%       18.7%           —
mINP            0.1%       

---
## 6. Summary table

In [15]:
print(f"{'Dataset':<14}{'encoder':<22}{'distance':<12}{'mAP':>8}{'Rank-1':>9}{'Rank-5':>9}{'Rank-10':>9}{'mINP':>8}")
print("-" * 91)


def _row(name, encoder, dist, m):
    return (
        f"{name:<14}{encoder:<22}{dist:<12}{m.map:>7.1f}%{m.rank1:>8.1f}%"
        f"{m.rank5:>8.1f}%{m.rank10:>8.1f}%{m.minp:>7.1f}%"
    )


print(_row("Market-1501", "OSNet", "cosine", result_market.metrics))
print(_row("", "", "euclidean", result_market_euc.metrics))
print(_row("MSMT17", "OSNet", "cosine", result_msmt17.metrics))
print(_row("", "", "euclidean", result_msmt17_euc.metrics))
print(_row("MSMT17", "FastReID MOT17", "cosine", result_fastreid_msmt17_cos.metrics))
print(_row("", "", "euclidean", result_fastreid_msmt17_euc.metrics))
print("-" * 91)
print("Model-zoo targets (OSNet, euclidean): Market-1501 R1≈94.2 mAP≈82.6  |  MSMT17 R1≈74.9 mAP≈43.8")
print("FastReID row is cross-domain (MOT17-trained encoder on MSMT17 gallery).")

Dataset       encoder               distance         mAP   Rank-1   Rank-5  Rank-10    mINP
-------------------------------------------------------------------------------------------
Market-1501   OSNet                 cosine         83.6%    94.2%    97.8%    98.6%   55.8%
                                    euclidean      82.5%    94.4%    97.8%    98.6%   51.0%
MSMT17        OSNet                 cosine         47.8%    75.9%    86.1%    89.3%    8.6%
                                    euclidean      43.7%    74.8%    85.1%    88.5%    5.3%
MSMT17        FastReID MOT17        cosine          2.1%     8.7%    15.7%    19.8%    0.1%
                                    euclidean       1.8%     8.1%    14.8%    18.7%    0.1%
-------------------------------------------------------------------------------------------
Model-zoo targets (OSNet, euclidean): Market-1501 R1≈94.2 mAP≈82.6  |  MSMT17 R1≈74.9 mAP≈43.8
FastReID row is cross-domain (MOT17-trained encoder on MSMT17 gallery).
